In [1]:
import pandas as pd
import numpy as np
import math
import glob
from scipy import ndimage as nd
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow
import cv2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Clase HandDP
clase para realizar el preprocesado de los datos de la base de datos. Se obtiene el template y los trazos realizados por el paciente.

In [97]:
class dataHandDP:
  def __init__(self, name_file):
    """
    @path: ruta de la imagen
    """
    self.path = f"/content/drive/MyDrive/{name_file}_out/"
    self.kernel_size = 5

  def preprocess_template(self, img):
    """
    @img: imagen a preprocesar
    @return: imagen preprocesada
    """
    return cv2.medianBlur(img,self.kernel_size)

  def preprocess_TH(self, img):
    """
    @img: imagen a preprocesar
    @return: imagen preprocesada
    """
    img_blur = cv2.blur(img, (self.kernel_size,self.kernel_size))
    return cv2.medianBlur(img_blur, self.kernel_size)

  def get_template(self, pixels):
    """
    @pixels: píxeles de la imagen
    @return: valor del píxel
    """
    #print("template",pixels)
    result = np.ones(3)*255
    if pixels[0] < 100 and pixels[1] < 100 and pixels[2] < 100:
      result = np.zeros(3)
    #
    return result

  def get_THimg(self, pixels):
    """
    @pixels: píxeles de la imagen
    @return: valor del píxel BGR
    """
    #print("th",pixels)
    result = pixels
    cond1 = pixels[0] - pixels[1]
    cond2 = pixels[0] - pixels[2]
    cond3 = pixels[1] - pixels[2]

    if abs(pixels[0] - pixels[1]) < 40 and abs(pixels[0] - pixels[2]) < 40 \
          and abs(pixels[1] - pixels[2]) < 40:
      result = np.ones(3)*255
    #print(result)
    #
    return result

  def save_img(self, id_img, imgTemplate, imgTH):
    """
    @imgTemplate: imagen preprocesada
    @imgTH: imagen preprocesada
    @return: imagen preprocesada
    """
    cv2.imwrite(f"{self.path}{id_img}_template.jpg", imgTemplate)
    cv2.imwrite(f"{self.path}{id_img}_TH.jpg", imgTH)

  def read_img(self, path):
    """
    @path: ruta de la imagen
    @return: imagen preprocesada
    """
    img = cv2.imread(path, cv2.COLOR_BGR2RGB)
    imgTemplate = self.preprocess_template(img.copy())
    imgTH = self.preprocess_TH(img.copy())
    # Recorrer píxeles
    for y in range(img.shape[0]):
        for x in range(img.shape[1]):
            pixel_template = self.get_template(imgTemplate[y, x])
            pixel_TH = self.get_THimg(imgTH[y, x])
            # Almacenar nuevos valores
            imgTemplate[y, x] = pixel_template
            imgTH[y, x] = pixel_TH
    #
    return imgTemplate, imgTH


  def run(self, path):
    """
    @path: ruta de la imagen
    @return: imagen preprocesada
    """
    print("ddd")
    for v_path in glob.glob(f"{path}*.jpg"):
      print("Process:", v_path)
      id_img = v_path.split("/")[-1].split(".")[0]
      imgTemplate, imgTH = self.read_img(v_path)
      self.save_img(id_img, imgTemplate, imgTH)


#Ejecución

In [ ]:
dp = dataHandDP("SpiralControl")
dp.run("/content/drive/MyDrive/SpiralControl/")